# PRMP Benchmark: PRMPConv vs Standard Aggregation with Ablations

This notebook demonstrates **Predictive Residual Message Passing (PRMPConv)** benchmarked against standard SAGEConv on bipartite relational graphs.

**Variants tested:**
1. **Standard**: SAGEConv baseline (mean aggregation)
2. **PRMP**: PRMPConv with learned predictions
3. **Random**: PRMPConv with random frozen predictions (ablation)
4. **No-subtract**: PRMPConv with concatenation instead of subtraction (ablation)

The demo generates synthetic bipartite graphs, trains all 4 GNN variants, and compares AUROC metrics against reference results from full-scale RelBench rel-hm experiments.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')
    _pip('torch==2.9.0', '--index-url', 'https://download.pytorch.org/whl/cpu')

# NOT pre-installed on Colab (needs torch installed first locally)
_pip('torch_geometric')

In [ ]:
import json
import os
import time
import math
import gc
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from torch_geometric.nn import MessagePassing, HeteroConv, SAGEConv
from torch_geometric.data import HeteroData
from sklearn.metrics import roc_auc_score, accuracy_score

import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter2_prmp_benchmark/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded reference data with keys: {list(data.keys())}")
print(f"FK links: {list(data['reference_results'].keys())}")
for link, info in data['reference_results'].items():
    gi = info['graph_info']
    print(f"  {link}: {gi['num_parents']} parents, {gi['num_children']} children, "
          f"mean_card={gi['mean_cardinality']:.1f}, R2={gi['cross_table_r2']:.4f}")
    print(f"    Variants: {list(info['variants'].keys())}")

## Configuration

All tunable hyperparameters are defined below. Values are set to small defaults for quick demo execution. Original full-scale values are noted in comments.

In [ ]:
# ── Hyperparameters (demo values; originals in comments) ──
HIDDEN_DIM = 32          # Original: 64
NUM_LAYERS = 2           # Original: 2
LEARNING_RATE = 0.001    # Original: 0.001
WEIGHT_DECAY = 1e-5      # Original: 1e-5
EPOCHS = 5               # Original: 50
SEED = 42                # Original: 42
VARIANTS = ['standard', 'prmp', 'random', 'no_subtract']

# ── Synthetic data parameters ──
NUM_PARENTS = 30         # Original: 629-11467
NUM_CHILDREN = 150       # Original: 200000
PARENT_FEAT_DIM = 5      # Original: 5-21
CHILD_FEAT_DIM = 3       # Original: 3

## PRMPConv Implementation

Predictive Residual Message Passing convolution (from the original `method.py`):
- 2-layer prediction MLP: `parent_feat -> predicted_child_feat`
- Residual = `child_feat - pred_mlp(parent_feat.detach())`
- LayerNorm on residuals
- Mean aggregation + GraphSAGE-style concat+linear update

In [ ]:
class PRMPConv(MessagePassing):
    """Predictive Residual Message Passing convolution.

    Per the PRMP architectural spec (Sections 3, 5, 7):
      - 2-layer prediction MLP: parent_feat -> predicted_child_feat
      - Residual = child_feat - pred_mlp(parent_feat.detach())
      - LayerNorm on residuals
      - Mean aggregation
      - GraphSAGE-style concat+linear update
    """

    def __init__(self, in_channels_src: int, in_channels_dst: int,
                 out_channels: int, mode: str = 'prmp'):
        super().__init__(aggr='mean')
        self.mode = mode
        self.in_channels_src = in_channels_src
        self.in_channels_dst = in_channels_dst
        self.out_channels = out_channels
        hidden = min(in_channels_dst, in_channels_src)

        # Prediction MLP: parent -> predicted child (Section 5.1)
        self.pred_mlp = nn.Sequential(
            nn.Linear(in_channels_dst, hidden),
            nn.ReLU(),
            nn.Linear(hidden, in_channels_src),
        )
        # Zero-initialize final layer (Section 5.3)
        nn.init.zeros_(self.pred_mlp[-1].weight)
        nn.init.zeros_(self.pred_mlp[-1].bias)

        # Random ablation: freeze with random weights
        if mode == 'random':
            nn.init.kaiming_normal_(self.pred_mlp[-1].weight)
            for p in self.pred_mlp.parameters():
                p.requires_grad = False

        # LayerNorm on residuals (Section 5.2, Option B)
        if mode != 'no_subtract':
            self.norm = nn.LayerNorm(in_channels_src)

        # GraphSAGE-style update (Section 3.4)
        if mode == 'no_subtract':
            self.update_mlp = nn.Linear(in_channels_dst + in_channels_src * 2, out_channels)
        else:
            self.update_mlp = nn.Linear(in_channels_dst + in_channels_src, out_channels)

    def forward(self, x, edge_index):
        if isinstance(x, torch.Tensor):
            x = (x, x)
        x_src, x_dst = x
        aggr_out = self.propagate(edge_index, x=x)
        out = self.update_mlp(torch.cat([x_dst, aggr_out], dim=-1))
        return out

    def message(self, x_j, x_i):
        """Compute residual messages.
        x_j = source (child) features, x_i = destination (parent) features.
        """
        predicted = self.pred_mlp(x_i.detach())
        if self.mode == 'no_subtract':
            return torch.cat([x_j, predicted], dim=-1)
        else:
            residual = x_j - predicted
            return self.norm(residual)

print("PRMPConv defined.")

## BipartiteGNN Model

Heterogeneous GNN for bipartite parent-child graphs supporting all 4 variants. Uses `HeteroConv` with child->parent edges (PRMPConv or SAGEConv) and parent->child reverse edges (always SAGEConv).

In [ ]:
class BipartiteGNN(nn.Module):
    """Heterogeneous GNN for bipartite parent-child graphs.
    4 variants: 'standard', 'prmp', 'random', 'no_subtract'.
    """

    def __init__(self, parent_feat_dim: int, child_feat_dim: int,
                 hidden_dim: int = 64, num_layers: int = 2,
                 conv_type: str = 'prmp'):
        super().__init__()
        self.conv_type = conv_type
        self.num_layers = num_layers

        self.parent_enc = nn.Sequential(
            nn.Linear(parent_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.child_enc = nn.Sequential(
            nn.Linear(child_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            if conv_type == 'standard':
                fwd_conv = SAGEConv((hidden_dim, hidden_dim), hidden_dim)
            else:
                fwd_conv = PRMPConv(hidden_dim, hidden_dim, hidden_dim, mode=conv_type)

            rev_conv = SAGEConv((hidden_dim, hidden_dim), hidden_dim)

            conv = HeteroConv({
                ('child', 'fk_to', 'parent'): fwd_conv,
                ('parent', 'rev_fk_to', 'child'): rev_conv,
            }, aggr='sum')

            self.convs.append(conv)
            self.norms.append(nn.ModuleDict({
                'parent': nn.LayerNorm(hidden_dim),
                'child': nn.LayerNorm(hidden_dim),
            }))

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x_dict, edge_index_dict):
        h_dict = {
            'parent': self.parent_enc(x_dict['parent']),
            'child': self.child_enc(x_dict['child']),
        }
        for i, conv in enumerate(self.convs):
            new_h = conv(h_dict, edge_index_dict)
            for node_type in h_dict:
                if node_type in new_h:
                    new_h[node_type] = self.norms[i][node_type](new_h[node_type])
                    new_h[node_type] = F.relu(new_h[node_type])
                    if i > 0:
                        new_h[node_type] = new_h[node_type] + h_dict[node_type]
                else:
                    new_h[node_type] = h_dict[node_type]
            h_dict = new_h
        return self.head(h_dict['parent']).squeeze(-1)

    @torch.no_grad()
    def get_prmp_diagnostics(self, x_dict, edge_index_dict):
        """Get PRMP-specific diagnostics: prediction R^2, residual stats."""
        if self.conv_type not in ('prmp', 'random', 'no_subtract'):
            return {}
        diagnostics = {}
        h_dict = {
            'parent': self.parent_enc(x_dict['parent']),
            'child': self.child_enc(x_dict['child']),
        }
        for layer_idx, conv in enumerate(self.convs):
            edge_type = ('child', 'fk_to', 'parent')
            if edge_type in conv.convs:
                prmp_conv = conv.convs[edge_type]
                if hasattr(prmp_conv, 'pred_mlp'):
                    ei = edge_index_dict[edge_type]
                    child_h = h_dict['child'][ei[0]]
                    parent_h = h_dict['parent'][ei[1]]
                    predicted = prmp_conv.pred_mlp(parent_h)
                    residual = child_h - predicted
                    ss_res = ((child_h - predicted) ** 2).sum().item()
                    ss_tot = ((child_h - child_h.mean(0)) ** 2).sum().item()
                    r2 = 1 - ss_res / max(ss_tot, 1e-8)
                    diagnostics[f'layer_{layer_idx}'] = {
                        'prediction_r2': round(r2, 6),
                        'residual_mean': round(residual.mean().item(), 6),
                        'residual_std': round(residual.std().item(), 6),
                        'residual_abs_mean': round(residual.abs().mean().item(), 6),
                        'pred_weight_norm': round(
                            sum(p.norm().item() for p in prmp_conv.pred_mlp.parameters()), 6),
                    }
            new_h = conv(h_dict, edge_index_dict)
            for nt in h_dict:
                if nt in new_h:
                    new_h[nt] = F.relu(new_h[nt])
                else:
                    new_h[nt] = h_dict[nt]
            h_dict = new_h
        return diagnostics

print("BipartiteGNN defined.")

## Synthetic Data Generation

Since the original experiment uses large RelBench parquet files, we generate synthetic bipartite graphs with similar structure for this demo.

In [ ]:
def generate_synthetic_graph(num_parents, num_children, parent_feat_dim, child_feat_dim, seed=SEED):
    """Generate a synthetic bipartite graph mimicking RelBench FK link structure."""
    np.random.seed(seed)
    torch.manual_seed(seed)

    parent_features = torch.randn(num_parents, parent_feat_dim)
    child_features = torch.randn(num_children, child_feat_dim)

    # Assign children to parents (varying cardinality)
    parent_indices = np.random.choice(num_parents, num_children).astype(np.int64)
    child_idx = np.arange(num_children, dtype=np.int64)

    edge_index_fwd = torch.from_numpy(np.stack([child_idx, parent_indices]))
    edge_index_rev = torch.from_numpy(np.stack([parent_indices, child_idx]))

    # Binary classification target correlated with parent features
    parent_signal = parent_features[:, 0] + 0.5 * parent_features[:, 1]
    targets = (parent_signal > parent_signal.median()).float()

    # Train/val/test split (70/15/15)
    pids = np.arange(num_parents)
    np.random.shuffle(pids)
    n_train = int(0.7 * num_parents)
    n_val = int(0.15 * num_parents)

    train_mask = torch.zeros(num_parents, dtype=torch.bool)
    val_mask = torch.zeros(num_parents, dtype=torch.bool)
    test_mask = torch.zeros(num_parents, dtype=torch.bool)
    train_mask[pids[:n_train]] = True
    val_mask[pids[n_train:n_train + n_val]] = True
    test_mask[pids[n_train + n_val:]] = True

    graph = HeteroData()
    graph['parent'].x = parent_features
    graph['parent'].y = targets
    graph['parent'].train_mask = train_mask
    graph['parent'].val_mask = val_mask
    graph['parent'].test_mask = test_mask
    graph['child'].x = child_features
    graph[('child', 'fk_to', 'parent')].edge_index = edge_index_fwd
    graph[('parent', 'rev_fk_to', 'child')].edge_index = edge_index_rev

    card_counts = np.bincount(parent_indices, minlength=num_parents)
    info = {
        'num_parents': num_parents,
        'num_children': num_children,
        'parent_feat_dim': parent_feat_dim,
        'child_feat_dim': child_feat_dim,
        'mean_cardinality': round(float(card_counts.mean()), 4),
        'positive_rate': round(float(targets.mean()), 4),
    }
    return {'data': graph, 'info': info}

graph_data = generate_synthetic_graph(NUM_PARENTS, NUM_CHILDREN, PARENT_FEAT_DIM, CHILD_FEAT_DIM)
print(f"Synthetic graph: {graph_data['info']['num_parents']} parents, "
      f"{graph_data['info']['num_children']} children, "
      f"mean_card={graph_data['info']['mean_cardinality']:.2f}, "
      f"pos_rate={graph_data['info']['positive_rate']:.2f}")

## Run Experiments

Train all 4 variants (standard, PRMP, random ablation, no-subtract ablation) on the synthetic bipartite graph and collect results.

In [ ]:
results = {}

for variant in VARIANTS:
    print(f"\n--- Training {variant} ---")
    graph = graph_data['data'].to(DEVICE)

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model = BipartiteGNN(
        parent_feat_dim=PARENT_FEAT_DIM,
        child_feat_dim=CHILD_FEAT_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        conv_type=variant,
    ).to(DEVICE)

    num_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Params: {num_params:,} total, {trainable_params:,} trainable")

    optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    train_losses = []
    val_aurocs = []
    best_val_auroc = 0.0
    best_epoch = 0
    best_state = None

    x_dict = {k: graph[k].x for k in ['parent', 'child']}
    edge_index_dict = {
        ('child', 'fk_to', 'parent'): graph[('child', 'fk_to', 'parent')].edge_index,
        ('parent', 'rev_fk_to', 'child'): graph[('parent', 'rev_fk_to', 'child')].edge_index,
    }

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        out = model(x_dict, edge_index_dict)
        mask = graph['parent'].train_mask
        loss = criterion(out[mask], graph['parent'].y[mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            out_val = model(x_dict, edge_index_dict)
            val_logits = out_val[graph['parent'].val_mask].cpu()
            val_targets = graph['parent'].y[graph['parent'].val_mask].cpu()
            val_probs = torch.sigmoid(val_logits).numpy()
            val_y = val_targets.numpy()
            try:
                val_auroc = roc_auc_score(val_y, val_probs)
            except ValueError:
                val_auroc = 0.5
        val_aurocs.append(val_auroc)

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch <= 3 or epoch == EPOCHS:
            print(f"  Epoch {epoch:3d}: loss={loss.item():.4f}, val_auroc={val_auroc:.4f}")

        if math.isnan(loss.item()):
            print(f"  NaN loss at epoch {epoch}, stopping")
            break

    train_time = time.time() - t0

    # Restore best model and evaluate on test set
    if best_state:
        model.load_state_dict(best_state)
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        out_test = model(x_dict, edge_index_dict)
        test_logits = out_test[graph['parent'].test_mask].cpu()
        test_targets = graph['parent'].y[graph['parent'].test_mask].cpu()
        test_probs = torch.sigmoid(test_logits).numpy()
        test_preds = (test_probs > 0.5).astype(int)
        test_y = test_targets.numpy()
        try:
            test_auroc = roc_auc_score(test_y, test_probs)
        except ValueError:
            test_auroc = 0.5
        test_acc = accuracy_score(test_y, test_preds)

    diagnostics = {}
    if variant in ('prmp', 'random', 'no_subtract'):
        try:
            diagnostics = model.get_prmp_diagnostics(x_dict, edge_index_dict)
        except Exception:
            pass

    results[variant] = {
        'test_auroc': round(test_auroc, 6),
        'test_accuracy': round(test_acc, 6),
        'best_val_auroc': round(best_val_auroc, 6),
        'best_epoch': best_epoch,
        'train_losses': train_losses,
        'val_aurocs': val_aurocs,
        'diagnostics': diagnostics,
        'train_time': round(train_time, 2),
    }
    print(f"  Test: AUROC={test_auroc:.4f}, Acc={test_acc:.4f}, Time={train_time:.1f}s")
    gc.collect()

print("\n=== All experiments complete ===")

## Results & Visualization

Compare demo results with reference results from the full-scale RelBench experiment.

In [ ]:
# ── Summary Table ──
print("=" * 70)
print(f"{'Variant':<15} {'Test AUROC':>12} {'Test Acc':>10} {'Best Val':>10} {'Time (s)':>10}")
print("-" * 70)
for v in VARIANTS:
    r = results[v]
    print(f"{v:<15} {r['test_auroc']:>12.4f} {r['test_accuracy']:>10.4f} "
          f"{r['best_val_auroc']:>10.4f} {r['train_time']:>10.2f}")
print("=" * 70)

# ── Reference results comparison ──
print("\nReference results from full-scale RelBench experiment:")
print("-" * 70)
for link_name, link_data in data['reference_results'].items():
    print(f"\n  {link_name} ({link_data['graph_info']['num_parents']} parents, "
          f"{link_data['graph_info']['num_children']} children):")
    for v in VARIANTS:
        if v in link_data['variants']:
            ref = link_data['variants'][v]['test_metrics']
            print(f"    {v:<15} AUROC={ref['auroc']:.4f}  Acc={ref['accuracy']:.4f}")

# ── Figure 1: AUROC bar chart (demo vs reference) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Demo results
demo_aurocs = [results[v]['test_auroc'] for v in VARIANTS]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
axes[0].bar(VARIANTS, demo_aurocs, color=colors, alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Test AUROC')
axes[0].set_title(f'Demo Results (synthetic, {NUM_PARENTS} parents, {EPOCHS} epochs)')
axes[0].set_ylim(0, 1.05)
for i, v in enumerate(demo_aurocs):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

# Reference results (first FK link)
first_link = list(data['reference_results'].keys())[0]
ref_data = data['reference_results'][first_link]
ref_aurocs = [ref_data['variants'][v]['test_metrics']['auroc'] for v in VARIANTS]
axes[1].bar(VARIANTS, ref_aurocs, color=colors, alpha=0.8, edgecolor='black')
axes[1].set_ylabel('Test AUROC')
axes[1].set_title(f'Reference: {first_link}')
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(ref_aurocs):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('auroc_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Figure 2: Training curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for v, c in zip(VARIANTS, colors):
    axes[0].plot(results[v]['train_losses'], label=v, color=c, linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Demo Training Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for v, c in zip(VARIANTS, colors):
    axes[1].plot(results[v]['val_aurocs'], label=v, color=c, linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation AUROC')
axes[1].set_title('Demo Validation AUROC Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

# ── PRMP Diagnostics ──
print("\nPRMP Diagnostics (demo):")
print("-" * 60)
for v in ['prmp', 'random', 'no_subtract']:
    diag = results[v].get('diagnostics', {})
    if diag:
        print(f"  {v}:")
        for layer, stats in sorted(diag.items()):
            print(f"    {layer}: R2={stats['prediction_r2']:.4f}, "
                  f"resid_std={stats['residual_std']:.4f}, "
                  f"weight_norm={stats['pred_weight_norm']:.4f}")

print("\nDone!")